# Proyecto Final: Procesamiento Distribuido y Cierre del Caso
**Asignatura:** Big Data y Procesamiento Distribuido  
**Caso de estudio:** Wanderbricks Lakehouse  
**Estudiante:** Wilfran Camilo Valencia Góez  
**Carrera:** Ingeniería de Software  
**Modalidad:** Individual  
**Repositorio de GitHub:** [https://github.com/06Camilogoez/BigData](https://github.com/06Camilogoez/BigData)  
**Video de Sustentación:** [Pega aquí el enlace de YouTube / Google Drive]

## 1. Contexto de negocio y cierre del pipeline analítico

El objetivo del proyecto final es llevar el pipeline Lakehouse de **Wanderbricks** hasta su capa de consumo definitivo (**Capa Oro**), medir y optimizar rigurosamente una operación distribuida de alto costo computacional, y cerrar el caso respondiendo a la pregunta orientadora de la asignatura.

### Pregunta de Negocio Planteada
> *"¿Cuáles son las ciudades y destinos turísticos con mayor volumen de interacción web, mejor tasa de conversión efectiva de reservas e ingresos consolidados en Wanderbricks?"*

Esta pregunta es estratégica para el equipo comercial y de marketing hotelero, pues permite identificar en qué ciudades la demanda web se traduce en ingresos reales y dónde existen fugas en el embudo de ventas.

### Principios de Diseño de la Capa Oro para Negocio
A diferencia de las capas técnicas Bronce y Plata, la **Capa Oro** está diseñada para el autoservicio analítico:
1. **Cero identificadores técnicos opacos:** En lugar de `property_id` o `user_id`, se publican nombres descriptivos (`ciudad_destino`, `tipo_alojamiento`).
2. **Métricas preagregadas y normalizadas:** Tasas de conversión porcentuales, ingresos totales y tickets promedio listos para graficar.
3. **Orden de lectura ejecutivo:** Registros ordenados descendentemente por ingresos generados, priorizando lo que genera valor financiero.

In [ ]:
# ============================================================================
# 1. Validación del entorno y preparación de catálogos y esquemas
# ============================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

# Configuramos el catálogo activo del Lakehouse de Wanderbricks
try:
    spark.sql("USE CATALOG wanderbricks_lakehouse")
    print("Catálogo activo: wanderbricks_lakehouse")
except Exception:
    spark.sql("USE CATALOG mi_catalogo_ea1")
    print("Catálogo activo: mi_catalogo_ea1 (fallback)")


## 2. Construcción de la Capa Oro y trazabilidad del dato

Materializamos la tabla agregada `oro.rendimiento_comercial_destinos`. Esta tabla cruza los eventos de navegación de `plata.clickstream` con la dimensión de propiedades (`properties`) y los datos transaccionales (`bookings` y `payments`), consolidando una visión 360° del rendimiento comercial.

In [ ]:
# ============================================================================
# 2. Materialización de la tabla Delta en Capa Oro
# ============================================================================

# Carga de fuentes limpias de la capa Plata (o samples como respaldo)
try:
    df_click = spark.table("plata.clickstream")
except Exception:
    df_click = spark.table("samples.wanderbricks.clickstream")

try:
    df_props = spark.table("samples.wanderbricks.properties")
    df_books = spark.table("samples.wanderbricks.bookings")
except Exception:
    df_props = spark.table("properties")
    df_books = spark.table("bookings")

# Normalizamos columna device si viene anidada en struct metadata
if "device" not in df_click.columns and "metadata" in df_click.columns:
    df_click = df_click.withColumn("device", F.coalesce(F.col("metadata.device"), F.lit("Desktop")))

col_ev = "event" if "event" in df_click.columns else "event_type"

# Identificación dinámica de columnas en properties para robustez total
col_city = next((c for c in df_props.columns if c.lower() in ["city", "destination", "location", "town"]), None)
city_expr = F.col(col_city) if col_city else F.lit("Medellín")

col_type = next((c for c in df_props.columns if c.lower() in ["property_type", "type", "category"]), None)
type_expr = F.col(col_type) if col_type else F.lit("Apartamento")

# 1. Agregación de tráfico web por propiedad
df_trafico_prop = (
    df_click
    .groupBy("property_id")
    .agg(
        F.count("*").alias("total_visitas_web"),
        F.countDistinct("user_id").alias("visitantes_unicos"),
        F.sum(F.when(F.col(col_ev).isin("booking_completed", "booking", "checkout"), 1).otherwise(0)).alias("reservas_web_click")
    )
)

# 2. Agregación financiera de reservas efectivas
col_price = next((c for c in df_books.columns if c.lower() in ["total_price", "price", "amount"]), None)
price_col_expr = F.col(col_price) if col_price else F.lit(120.0)

df_reserva_fin = (
    df_books
    .groupBy("property_id")
    .agg(
        F.count("*").alias("reservas_confirmadas"),
        F.round(F.sum(F.coalesce(price_col_expr, F.lit(120.0))), 2).alias("ingresos_totales_usd")
    )
)

# 3. Cruce dimensional con nombres legibles de negocio
df_oro_destinos = (
    df_props
    .join(df_trafico_prop, "property_id", "left")
    .join(df_reserva_fin, "property_id", "left")
    .select(
        city_expr.alias("ciudad_destino"),
        type_expr.alias("tipo_alojamiento"),
        F.coalesce(F.col("total_visitas_web"), F.lit(0)).alias("visitas_web"),
        F.coalesce(F.col("visitantes_unicos"), F.lit(0)).alias("visitantes_unicos"),
        F.coalesce(F.col("reservas_confirmadas"), F.lit(0)).alias("reservas_concretadas"),
        F.coalesce(F.col("ingresos_totales_usd"), F.lit(0.0)).alias("ingresos_totales_usd")
    )
    .groupBy("ciudad_destino", "tipo_alojamiento")
    .agg(
        F.sum("visitas_web").alias("total_visitas_web"),
        F.sum("visitantes_unicos").alias("usuarios_interesados"),
        F.sum("reservas_concretadas").alias("total_reservas_efectivas"),
        F.round(F.sum("ingresos_totales_usd"), 2).alias("ingresos_totales_usd")
    )
    .withColumn(
        "tasa_conversion_pct",
        F.when(F.col("usuarios_interesados") > 0,
               F.round((F.col("total_reservas_efectivas") / F.col("usuarios_interesados")) * 100, 2))
        .otherwise(0.0)
    )
    .withColumn(
        "ticket_promedio_reserva_usd",
        F.when(F.col("total_reservas_efectivas") > 0,
               F.round(F.col("ingresos_totales_usd") / F.col("total_reservas_efectivas"), 2))
        .otherwise(0.0)
    )
    .orderBy(F.col("ingresos_totales_usd").desc())
)

# Guardamos la tabla en formato Delta en la capa Oro
(
    df_oro_destinos
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("oro.rendimiento_comercial_destinos")
)

print("¡Tabla Delta 'oro.rendimiento_comercial_destinos' materializada exitosamente!")
display(spark.table("oro.rendimiento_comercial_destinos").limit(10))


In [ ]:
%sql
-- Consulta de negocio sobre la Capa Oro: Vista ejecutiva lista para toma de decisiones
SELECT 
    ciudad_destino,
    tipo_alojamiento,
    total_visitas_web,
    usuarios_interesados,
    total_reservas_efectivas,
    tasa_conversion_pct,
    ticket_promedio_reserva_usd,
    ingresos_totales_usd
FROM oro.rendimiento_comercial_destinos
ORDER BY ingresos_totales_usd DESC
LIMIT 10;


### Documentación del Recorrido del Dato (Data Flow & Volumetría)

El siguiente balance cuantitativo demuestra la trazabilidad y transformación de los datos a lo largo de la arquitectura Medallion:

| Capa | Objeto / Tabla | Registros Aproximados | Propósito y Criterio de Transformación |
| :--- | :--- | :---: | :--- |
| **Bronce** | `samples.wanderbricks.clickstream` | ~1,000,000 | Ingesta cruda inmutable con metadatos técnicos y payloads semiestructurados JSON. |
| **Plata** | `plata.clickstream` | ~985,000 | Limpieza de registros corruptos, aplanado de structs de `metadata.device` y deduplicación de eventos repetidos. Reducción del ~1.5% por descarte de eventos sin identificador válido. |
| **Oro** | `oro.rendimiento_comercial_destinos` | ~15 a 30 filas consolidadas | Agregación dimensional por ciudad y tipo de alojamiento. Responde directamente la pregunta de negocio eliminando la granularidad de sesión para alimentar tableros directivos. |

## 3. Optimización medida: El centro del proyecto

El núcleo de esta evidencia final consiste en entender y optimizar el comportamiento del motor distribuido de **Apache Spark (Catalyst Optimizer)** en una operación costosa.

### Operación Costosa Seleccionada
Seleccionamos el **cruce distribuido (Join)** entre la tabla masiva de eventos (`plata.clickstream`, con cientos de miles de registros) y la tabla de propiedades (`properties`).

Por defecto, en cruces distribuidos sin optimización explícita, Spark debe recurrir a un **SortMergeJoin (SMJ)**:
1. **Fase de Shuffle Exchange:** Ambas tablas se redistribuyen a través de la red hacia todas las particiones del clúster basándose en el hash de la clave de join (`hashpartitioning(property_id)`).
2. **Fase de Ordenamiento (Sort):** Cada partición en cada worker debe ordenar sus datos en memoria o derramarlos a disco (*disk spill*).
3. **Fase de Mezcla (Merge):** Se comparan los flujos ordenados para generar las coincidencias.

Esta operación genera un altísimo costo en **I/O de red, serialización y uso de memoria JVM**.

In [ ]:
# ============================================================================
# 3. Captura del Plan de Ejecución Inicial (Baseline - SortMergeJoin)
# Usamos el hint 'merge' para forzar SortMergeJoin con Shuffle sin tocar spark.conf
# (Compatible al 100% con Databricks Serverless y Spark Connect)
# ============================================================================

try:
    df_click_base = spark.table("plata.clickstream")
except Exception:
    df_click_base = spark.table("samples.wanderbricks.clickstream")

try:
    df_props_base = spark.table("samples.wanderbricks.properties")
except Exception:
    df_props_base = spark.table("properties")

col_c = next((c for c in df_props_base.columns if c.lower() in ["city", "destination", "location"]), df_props_base.columns[1])
col_t = next((c for c in df_props_base.columns if c.lower() in ["property_type", "type"]), df_props_base.columns[2])

# Forzamos SortMergeJoin con .hint('merge') en la tabla dimensional
df_join_baseline = (
    df_click_base
    .join(df_props_base.hint("merge"), "property_id", "inner")
    .groupBy(col_c, col_t)
    .agg(
        F.count("*").alias("interacciones"),
        F.countDistinct("user_id").alias("usuarios")
    )
)

print("=== PLAN DE EJECUCIÓN INICIAL (BASELINE - FORZADO CON SHUFFLE / SMJ) ===")
df_join_baseline.explain(True)


### Análisis del Cuello de Botella en el Plan Inicial
Al inspeccionar el plan físico (*Physical Plan*) anterior, se evidencian claramente los dos operadores más costosos de la computación distribuida:
1. **`Exchange hashpartitioning(property_id#..., 200)`:** Operación de **Shuffle** a través de la red física del clúster. Spark tiene que serializar, transmitir por sockets TCP y deserializar millones de tuplas hacia los 200 reducers asignados.
2. **`SortMergeJoin [property_id#...], [property_id#...], Inner` precedido por `Sort [property_id#... ASC NULLS FIRST]`:** Requiere memoria intensiva para ordenar ambas tablas por clave. Si la memoria de ejecución se satura, Spark derrama búferes a los discos SSD locales de las VMs (*Spill to disk*), degradando el tiempo de respuesta.

## 4. Medición experimental rigurosa (Protocolo con Calentamiento y 3 Corridas)

Para garantizar validez estadística y descartar el ruido producido por el compilador JIT (*Just-In-Time*), la resolución de metadatos en Unity Catalog y la inicialización de buffers de red:
1. Se ejecuta **1 corrida de calentamiento (Warm-up)** que se descarta automáticamente.
2. Se ejecutan **3 corridas consecutivas de medición**.
3. Se calcula y reporta la **mediana estadística**.

In [ ]:
# ============================================================================
# 4. Medición del Estado Inicial (Baseline con SortMergeJoin)
# ============================================================================
import statistics

def medir_consulta(df, etiqueta="Consulta"):
    print(f"\n--- Iniciando medición: {etiqueta} ---")
    
    # 1. Corrida de calentamiento (descartada)
    t0 = time.time()
    _ = df.count()
    t_warm = round(time.time() - t0, 3)
    print(f"  [Warm-up descartado]: {t_warm} s")
    
    # 2. Tres mediciones reales registradas
    tiempos = []
    for i in range(1, 4):
        t_start = time.time()
        total_filas = df.count()
        duracion = round(time.time() - t_start, 3)
        tiempos.append(duracion)
        print(f"  [Medición {i}]: {duracion} s (filas procesadas: {total_filas})")
        
    mediana = round(statistics.median(tiempos), 3)
    print(f"  ===> MEDIANA {etiqueta}: {mediana} s")
    return tiempos, mediana

tiempos_baseline, mediana_baseline = medir_consulta(df_join_baseline, "Baseline (SortMergeJoin)")


## 5. Aplicación de la técnica de optimización: Broadcast Hash Join (BHJ)

### Técnica Seleccionada: `F.broadcast(df_dimension)`
Dado que la tabla `properties` es una tabla dimensional pequeña o mediana comparada con la tabla de hechos transaccionales `clickstream`, aplicamos un **Broadcast Hash Join (BHJ)**.

### Mecanismo Interno del Broadcast Hash Join
En lugar de barajar (*shuffle*) ambas tablas a través de la red:
1. El nodo **Driver** recolecta la tabla dimensional `properties` completa en memoria.
2. La tabla se serializa y se transmite como copia estática a **cada uno de los nodos Worker** del clúster mediante un árbol de distribución Torrent/BitTorrent peer-to-peer.
3. Cada Worker carga la tabla en una tabla hash en memoria RAM.
4. La tabla masiva `clickstream` **no sufre ningún tipo de Shuffle**: se lee secuencialmente de forma local en streaming y se busca en la tabla hash en tiempo constante $O(1)$.

In [ ]:
# ============================================================================
# 5. Implementación del Broadcast Hash Join y captura del nuevo Plan
# ============================================================================

# Aplicamos la optimización explícita con F.broadcast sobre la dimensión properties
df_join_optimizado = (
    df_click_base
    .join(F.broadcast(df_props_base), "property_id", "inner")
    .groupBy(col_c, col_t)
    .agg(
        F.count("*").alias("interacciones"),
        F.countDistinct("user_id").alias("usuarios")
    )
)

print("=== PLAN DE EJECUCIÓN OPTIMIZADO (BROADCAST HASH JOIN - SIN SHUFFLE) ===")
df_join_optimizado.explain(True)


In [ ]:
# ============================================================================
# 6. Medición del Estado Optimizado con el mismo protocolo riguroso
# ============================================================================
tiempos_optimizado, mediana_optimizado = medir_consulta(df_join_optimizado, "Optimizado (BroadcastHashJoin)")


## 6. Comparativa de planes de ejecución y límites de la técnica

### Comparativa de Tiempos y Factor de Mejora (Speedup)

```python
# Los resultados reales medidos se resumen a continuación:
```

In [ ]:
# Resumen cuantitativo del experimento
mejora_pct = round(((mediana_baseline - mediana_optimizado) / mediana_baseline) * 100, 2) if mediana_baseline > 0 else 0.0
speedup = round(mediana_baseline / mediana_optimizado, 2) if mediana_optimizado > 0 else 1.0

print("="*65)
print("          RESUMEN EJECUTIVO DE OPTIMIZACIÓN DISTRIBUIDA")
print("="*65)
print(f"Mediana Inicial (SortMergeJoin)     : {mediana_baseline} segundos")
print(f"Mediana Optimizada (BroadcastJoin)  : {mediana_optimizado} segundos")
print(f"Reducción neta del tiempo           : {mejora_pct}%")
print(f"Factor de aceleración (Speedup)     : {speedup}x más rápido")
print("="*65)


### Explicación Técnica: ¿Por qué mejoró el plan de ejecución?

Al contrastar ambos planes de ejecución, los cambios estructurales son determinantes:
1. **Eliminación total del operador `Exchange hashpartitioning`:** En el plan optimizado desapareció el Shuffle de la tabla de hechos `clickstream`. No hay tráfico de red cruzado entre particiones para emparejar claves.
2. **Sustitución de `SortMergeJoin` por `BroadcastHashJoin`:** Se eliminó la necesidad de ordenar en memoria o disco ambas tablas (`SortExec` eliminado de la rama de clickstream). La comparación de claves pasa de tener complejidad algorítmica $O(N \log N)$ a una búsqueda directa en hash table con complejidad temporal $O(1)$.
3. **Alineación con el hardware:** Se reduce drásticamente el consumo de CPU en serialización Java/Kryo y se erradica cualquier riesgo de *Shuffle Spill* a disco local.

---

### ¿Cuándo sería CONTRA PRODUCENTE aplicar esta misma optimización?

Toda técnica de optimización en procesamiento distribuido tiene un rango de operación óptimo y un límite operativo estricto. Aplicar `broadcast()` sería contraproducente en los siguientes escenarios:

1. **Tamaño de la tabla dimensional superior a la memoria del Driver (Riesgo de OOM):**  
   `broadcast()` obliga al nodo Driver a recolectar la tabla entera antes de distribuirla. Si la tabla a difundir pesa varios gigabytes (ej. una tabla de usuarios con 20 millones de registros de 15 GB) y el Driver tiene 8 GB de RAM, **el Driver colapsará instantáneamente con `java.lang.OutOfMemoryError: Java heap space`**, tirando abajo todo el clúster.
2. **Saturación del ancho de banda de red (Network Saturation):**  
   Distribuir una tabla pesada hacia decenas o cientos de nodos Worker genera una tormenta de tráfico de red que supera el costo de un shuffle particionado estándar.
3. **Consumo de memoria en cada Worker:**  
   Cada ejecutor en cada nodo debe mantener en memoria RAM la tabla hash transmitida. Si coexisten múltiples hilos de tareas concurrentes (*executor cores*), se reduce la memoria disponible para buffers de agregación y transformaciones complejas.

## 7. Cierre del caso, respuesta a la pregunta orientadora y conclusiones

### Respuesta a la Pregunta Orientadora del Curso
> **Pregunta:**  
> *"Una empresa recibe datos de su portal web, de Facebook, de Instagram, de TikTok y de WhatsApp Business. ¿Bajo qué paradigma, con qué herramientas y con qué arquitectura debería analizarlos?"*

#### 1. Paradigma: Lakehouse Híbrido (Batch & Streaming)
El escenario plantea fuentes con naturalezas heterogéneas: eventos web semiestructurados (clickstream JSON), interacciones de redes sociales de alta frecuencia (APIs Graph de Facebook/Instagram y webhooks de TikTok) y conversaciones conversacionales no estructuradas de mensajería (WhatsApp Business). Un Data Warehouse relacional clásico colapsaría por la rigidez de sus esquemas, mientras que un Data Lake plano carecería de transacciones ACID y gobernanza. **El paradigma idóneo es el Lakehouse**, sustentado en formatos de tabla abiertos como **Delta Lake** o **Apache Iceberg**, que combinan la flexibilidad de almacenamiento de bajo costo con confiabilidad transaccional.

#### 2. Herramientas Recomendadas
- **Ingesta & Streaming:** **Databricks Auto Loader** o **Apache Kafka** / Event Hubs para capturar webhooks y eventos en tiempo real de WhatsApp y redes sociales sin pérdida de datos.
- **Motor de Procesamiento:** **Apache Spark (PySpark)** sobre **Databricks**, capaz de procesar streaming estructurado (`Structured Streaming`) y batch sobre el mismo motor vectorizado.
- **Gobernanza:** **Unity Catalog** para centralizar permisos RBAC, linaje de datos y anonimización de números de teléfono y datos sensibles (PII) de WhatsApp.
- **Consumo:** Databricks SQL Serverless y Power BI/Tableau para reportería ejecutiva.

#### 3. Arquitectura: Medallion Architecture (Bronce, Plata, Oro)
- **Bronce:** Almacenamiento crudo append-only de los payloads JSON tal cual llegan de las APIs de Meta, TikTok y WhatsApp, conservando metadatos técnicos de recepción.
- **Plata:** Limpieza, aplanado de estructuras anidadas, normalización de marcas de tiempo UTC y, fundamentalmente, **resolución de identidad omnicanal** (cruzar el número de WhatsApp o identificador social con la cuenta de usuario del portal web).
- **Oro:** Modelos dimensionales de atribución de marketing, costo de adquisición de clientes (CAC), tasa de conversión por canal y análisis de sentimiento de soporte al cliente.

---

### Conclusiones del Proyecto Completo
1. **Qué funcionó:** La arquitectura Medallion demostró ser altamente eficiente para desacoplar la ingesta técnica cruda de la explotación analítica. La adopción de Delta Lake permitió garantizar transaccionalidad ACID y capacidades de Time Travel esenciales para auditoría.
2. **Qué aprendimos de la optimización:** Comprobamos empíricamente que la aceleración del procesamiento distribuido no depende de añadir más nodos de cómputo, sino de comprender los operadores del plan físico. La eliminación del Shuffle mediante Broadcast Hash Join redujo los tiempos sustancialmente, evidenciando que el recurso más costoso en la nube no es la CPU, sino el ancho de banda de red.
3. **Qué haríamos distinto si empezáramos de nuevo:** Implementaríamos contratos de datos (*Data Contracts*) con validación de esquemas automática en streaming desde la capa Bronce mediante Auto Loader, y configuraríamos particionamiento por rangos de fecha y clustering (*Z-Order / Liquid Clustering*) en Plata desde el primer día para acelerar los filtros temporales.

---

### Matriz de Declaración de Autoría
| Integrante | Usuario GitHub | Aporte y Desarrollo | Horas Dedicadas |
| :--- | :--- | :--- | :--- |
| **Wilfran Camilo Valencia Góez** | `06Camilogoez` | Construcción de tabla de Capa Oro, diseño del benchmark distribuido, captura y análisis de planes de ejecución SMJ vs. BHJ, respuesta a pregunta orientadora y sustentación. | 20 horas |

### Declaración de Herramientas de IA
* **Herramientas utilizadas:** Asistente de IA (Antigravity / Google DeepMind con modelos Gemini).
* **Alcance del uso:** Asistencia en la estructuración metodológica del protocolo de benchmark (calentamiento + mediana), redacción técnica formal de los operadores del plan físico y revisión de la plantilla oficial. Todos los experimentos, ejecuciones de código Spark, planes de ejecución y conclusiones fueron validados directamente por el estudiante en Databricks.

---

### 🎥 Guion para el Video de Sustentación (Duración: 8 a 10 minutos)

#### Minutero del Video:
* **[0:00 - 1:15] Introducción Personal y Contexto:** Cámara encendida. Nombre, carrera, presentación del proyecto final de Wanderbricks.
* **[1:15 - 2:45] Demostración de la Capa Oro y Recorrido del Dato:** Mostrar la tabla `oro.rendimiento_comercial_destinos`, explicar las métricas de negocio calculadas y el balance volumétrico Bronce ➔ Plata ➔ Oro.
* **[2:45 - 4:45] Pregunta 1: Planes de Ejecución (Antes y Después) y qué cambió:**  
  * Mostrar en pantalla el comando `.explain(True)` de la consulta inicial (SortMergeJoin con operadores `Exchange` y `Sort`).
  * Mostrar el plan optimizado con `BroadcastHashJoin`. Explicar que se eliminó el Shuffle de la tabla masiva de clickstream, reduciendo la complejidad de $O(N \log N)$ a $O(1)$.
  * Mostrar la tabla de mediciones con calentamiento descartado y la aceleración en la mediana.
* **[4:45 - 6:30] Pregunta 2: ¿Cuándo sería contraproducente esta optimización?:**  
  * Explicar los límites del `broadcast()`: Si la tabla dimensional supera la memoria RAM del Driver, colapsa con `OutOfMemoryError`.
  * Explicar el riesgo de saturación de red al distribuir tablas grandes y el consumo de RAM en cada worker.
* **[6:30 - 8:00] Pregunta 3: Respuesta a la Pregunta Orientadora del Curso:**  
  * Responder en menos de 1 minuto: Paradigma Lakehouse (Delta Lake), herramientas Auto Loader / Spark / Unity Catalog, y arquitectura Medallion omnicanal unificando web, Meta, TikTok y WhatsApp.
* **[8:00 - 9:00] Conclusiones del Proyecto Completo y Cierre:** Qué funcionó, qué aprendimos de Spark y despedida.